# Demo 2: First evaluation

**Concept**: `mlflow.genai.evaluate()` runs your agent over every row of a dataset, then scores each
output with **scorers**. It has three ingredients:

```
data  (inputs + expectations) ---+
scorers (built-in judges)     ---+--> mlflow.genai.evaluate() --> results (aggregate + per row)
predict_fn (our traced agent) ---+
```

Why demo 1 came first: evaluation runs the agent, creates a trace per row, and some judges (e.g.
`RetrievalGroundedness`) read the **retriever span** of that trace.

Note: this uses `mlflow.genai.evaluate()`. The older `mlflow.evaluate` is deprecated for LLM apps.

In [0]:
%pip install -U "mlflow[databricks]>=3.1" openai databricks-sdk
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 167.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 163.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 88.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 164.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 118.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.6/771.6 kB 38.1 MB/s  0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-47e4c415-edad-47e4-aa94-7860e1e35cb1
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: idna
    Found e

In [0]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import mlflow

from common.config import EXPERIMENT_PATH

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_PATH)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1571690280311094', creation_time=1792292632487, effective_trace_archival_retention=None, experiment_id='1571690280311094', last_update_time=1792292632487, lifecycle_stage='active', name='/Shared/python-wiki-agent-experiment', tags={'mlflow.experiment.sourceName': '/Shared/python-wiki-agent-experiment',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'pranav@platformatory.com',
 'mlflow.ownerId': '77162250982329'}, trace_location=None, workspace='default'>

## 1. The dataset

Each row has `inputs` (keys match the `predict_fn` arguments) and `expectations` (ground truth). Our 10 questions include: 6 covered by the wiki, 2 **not** in the wiki, 1 out of scope, 1 adversarial.

In [0]:
from common.eval_data import EVAL_DATA, KIND_BY_QUESTION

print(f"{len(EVAL_DATA)} rows. Example row:\n{EVAL_DATA[0]}")

10 rows. Example row:
{'inputs': {'question': 'How do I add an item to a list?'}, 'expectations': {'expected_facts': ['Use the append() method, e.g. nums.append(4)']}}


## 2. The predict function

Rules for `predict_fn`: accept the keys of `inputs` as keyword arguments, return something JSON-serialisable, be traced, and emit exactly one trace per call. Our `answer_question` already does all of that, so the wrapper is trivial.

In [0]:
from common.agent_traced import answer_question


def predict_fn(question: str) -> dict:
    return answer_question(question)


## 3. Choose scorers (built-in LLM judges)

| Judge | Question it answers | Needs |
|---|---|---|
| `Correctness` | Does the response match the expected facts? | `expected_facts` |
| `Safety` | Is the response free of harmful content? | nothing |
| `RetrievalGroundedness` | Is the response supported by what was retrieved? | a `RETRIEVER` span in the trace |

Judges are LLMs. By default they use a Databricks-hosted judge model. To pick your own, pass `model="databricks:/<endpoint-name>"` to each scorer.

In [0]:
from mlflow.genai.scorers import Correctness, RetrievalGroundedness, Safety

scorers = [Correctness(), Safety(), RetrievalGroundedness()]

## 4. Run the evaluation

One call runs the agent on every row, creates a trace per row, and applies all scorers.

In [0]:
results = mlflow.genai.evaluate(
    data=EVAL_DATA,
    predict_fn=predict_fn,
    scorers=scorers,
)

print(f"run_id: {results.run_id}")
results.metrics

2026/10/18 03:34:58 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/18 03:34:58 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

run_id: ffa20ece50b840e2be9684f7acc04efc


{'correctness/mean': np.float64(0.8),
 'retrieval_groundedness/mean': np.float64(0.5),
 'safety/mean': np.float64(1.0)}

## 5. What to look at

Open the **Evaluations** tab of the experiment (the run link printed above). For each row, every judge returns a **`Feedback`** object with:

- `value`: pass/fail (or a score)
- `rationale`: *why*. Read these. They tell you what failed, not just that it failed.
- `source`: `LLM_JUDGE` here (`CODE` for `@scorer` functions, which is demo 3)

Click into a row's trace to jump from a failing score to the exact span that caused it.

Things to check:
1. Which of the 10 questions fail `RetrievalGroundedness`? (Likely the two **not_in_wiki** rows, but LLM judges vary, so check.)
2. Do those same rows still *pass* `Correctness`? Correct but ungrounded is exactly what demo 1 hinted at.
3. What does `Safety` say about the adversarial prompt?

In [0]:
# Optional: the same data as a table, grouped by question kind. Handy for slicing results.
# Best-effort: column names returned by `search_traces` can differ across MLflow versions, so adjust "request" / "assessments" below if this cell errors. The UI shows the same data.
import pandas as pd

traces = mlflow.search_traces(run_id=results.run_id)
rows = []
for _, t in traces.iterrows():
    question = t["request"]["question"] if isinstance(t["request"], dict) else str(t["request"])
    for a in t["assessments"]:
        rows.append(
            {
                "kind": KIND_BY_QUESTION.get(question, "?"),
                "question": question,
                "judge": a["assessment_name"],
                "value": a["feedback"]["value"] if a.get("feedback", False) else None,
            }
        )

display(pd.DataFrame(rows).pivot_table(index=["kind", "question"], columns="judge", values="value", aggfunc="first"))

correctness,retrieval_groundedness,safety
yes,no,yes
no,no,yes
yes,yes,yes
yes,yes,yes
yes,yes,yes
yes,yes,yes
yes,no,yes
yes,yes,yes
yes,no,yes
no,no,yes


## 6. Break it

Our tutor should give **beginners a runnable code example** with each answer. Look at the judge list again:  no built-in judge checks that. `Correctness` can pass an answer that explains the idea in prose only.

Built-in judges cover common criteria. Your product has *specific* ones.

**Next: demo 3 adds custom scorers (a guideline judge, a deterministic `@scorer`, and `make_judge()`) to check this and run them all in a single `evaluate()` call.**